# KISTEP 전체브리프 웹크롤링 (정적 페이지)

- 대상: KISTEP 브리프 > 전체브리프 목록
- URL: https://www.kistep.re.kr/board.es?mid=a10306010000&bid=0031
- 페이지 이동 파라미터: `nPage`
- 수집 페이지: 1 ~ 3 page
- 추출 정보: 카테고리 / 제목 / 저자 / 등록일 / 상세페이지 URL / 다운로드 URL / 미리보기 URL / 썸네일 URL

In [1]:
import time                       # 요청 사이 간격을 주기 위한 모듈
from datetime import datetime      # 수집 시각 기록
from urllib.parse import urljoin   # 상대주소를 완전한 URL로 변환

import requests                    # 웹페이지에 요청을 보내고 응답을 받는 라이브러리
from bs4 import BeautifulSoup      # HTML을 태그 단위로 탐색하는 라이브러리
import pandas as pd                # 표 형태로 정리/저장하는 라이브러리

# 목록 페이지 기본 주소 (페이지 이동은 &nPage= 로 처리)
LIST_URL = "https://www.kistep.re.kr/board.es?mid=a10306010000&bid=0031"

# 상대주소 앞에 붙일 사이트 기본 주소
BASE_URL = "https://www.kistep.re.kr/"

# 평범한 브라우저 요청처럼 보이도록 User-Agent 지정
headers = {"User-Agent": "Mozilla/5.0"}

print("준비 완료")

준비 완료


## 1) 한 페이지를 수집하는 함수 정의

`nPage` 파라미터로 페이지를 이동하며, 게시물(`li`) 하나마다 7개 정보를 추출합니다.
- 다운로드 링크: `.btn_icon a` 의 1번째 (`/boardDownload.es?...`)
- 미리보기 링크: `.btn_icon a` 의 2번째 (`/boardSynapPreview.es?...`)
- 썸네일 이미지: `.thumb img` 의 `src` (`/boardImgView.es?...`)

In [2]:
def crawl_page(page):
    """page 번호 하나를 받아 그 페이지 게시물 목록을 수집해 리스트로 반환."""
    page_url = f"{LIST_URL}&nPage={page}"      # 페이지 이동 파라미터 nPage 적용
    print("요청 URL:", page_url)

    response = requests.get(page_url, headers=headers)   # HTML 요청
    response.raise_for_status()                          # 실패 시 오류 발생

    soup = BeautifulSoup(response.text, "html.parser")   # 탐색 가능한 형태로 변환
    posts = soup.select(".publication_list > li")        # 게시물 1건 = li 1개

    page_items = []

    for post in posts:
        # 각 요소를 게시물(post) 범위 안에서 찾음
        category_el = post.select_one(".status")                           # 카테고리
        title_el = post.select_one(".item .title a")                       # 제목 + 상세 링크
        author_el = post.select_one(".basic_info li:nth-of-type(1) .txt")  # 저자
        date_el = post.select_one(".basic_info li:nth-of-type(2) .txt")    # 등록일
        download_el = post.select_one(".btn_icon a:nth-of-type(1)")        # 다운로드 링크
        preview_el = post.select_one(".btn_icon a:nth-of-type(2)")         # 미리보기 링크
        thumbnail_el = post.select_one(".thumb img")                       # 썸네일 이미지

        # 요소가 없을 때 오류가 나지 않도록 None 처리
        category = category_el.get_text(strip=True) if category_el else None
        title = title_el.get_text(strip=True) if title_el else None
        author = author_el.get_text(strip=True) if author_el else None
        published_date = date_el.get_text(strip=True) if date_el else None
        detail_url = urljoin(BASE_URL, title_el["href"]) if title_el else None
        download_url = urljoin(BASE_URL, download_el["href"]) if download_el else None
        preview_url = urljoin(BASE_URL, preview_el["href"]) if preview_el else None
        thumbnail_url = urljoin(BASE_URL, thumbnail_el["src"]) if thumbnail_el else None

        page_items.append({
            "category": category,            # 카테고리
            "title": title,                  # 제목
            "author": author,                # 저자
            "published_date": published_date,  # 등록일
            "detail_url": detail_url,        # 상세페이지 URL
            "download_url": download_url,    # 다운로드 URL
            "preview_url": preview_url,      # 미리보기 URL
            "thumbnail_url": thumbnail_url,  # 썸네일 URL
            "page_no": page,                 # 수집된 페이지 번호
        })

    return page_items


# 1페이지로 동작 확인
test_items = crawl_page(1)
print("수집 건수:", len(test_items))
test_items[0]

요청 URL: https://www.kistep.re.kr/board.es?mid=a10306010000&bid=0031&nPage=1
수집 건수: 10


{'category': '정책브리프',
 'title': '동물실험 대체 전환기, 오가노이드 현황과 시사점 - 주요국 정책 및 국가연구개발사업을 중심으로 -',
 'author': 'KISTEP 바이오혁신전략팀 심현아 부연구위원, 윤희정 팀장',
 'published_date': '2026-06-08',
 'detail_url': 'https://www.kistep.re.kr/board.es?mid=a10306010000&bid=0031&b_list=10&act=view&list_no=94746&nPage=1&keyField=&orderby=',
 'download_url': 'https://www.kistep.re.kr/boardDownload.es?bid=0031&list_no=94746&seq=1',
 'preview_url': 'https://www.kistep.re.kr/boardSynapPreview.es?bid=0031&list_no=94746&seq=1',
 'thumbnail_url': 'https://www.kistep.re.kr/boardImgView.es?bid=0031&list_no=94746',
 'page_no': 1}

## 2) 1 ~ 3 페이지 전체 수집

In [3]:
num_pages = 3       # 수집할 페이지 수 (1 ~ 3 page)

items = []          # 모든 페이지 결과를 모을 리스트

for page in range(1, num_pages + 1):     # 1페이지부터 차례로
    print(f"{page}페이지 수집 중...")
    page_items = crawl_page(page)        # 한 페이지 수집
    items.extend(page_items)             # 결과 합치기
    time.sleep(1)                        # 서버 부담을 줄이려 1초 대기

print("전체 수집 건수:", len(items))

1페이지 수집 중...
요청 URL: https://www.kistep.re.kr/board.es?mid=a10306010000&bid=0031&nPage=1
2페이지 수집 중...
요청 URL: https://www.kistep.re.kr/board.es?mid=a10306010000&bid=0031&nPage=2
3페이지 수집 중...
요청 URL: https://www.kistep.re.kr/board.es?mid=a10306010000&bid=0031&nPage=3
전체 수집 건수: 30


## 3) 표로 정리하고 CSV로 저장

In [ ]:
df = pd.DataFrame(items)

# 참조 테이블(kistep_briefs_vibe.csv)과 동일한 컬럼 구성
df["year"] = pd.to_datetime(df["published_date"]).dt.year
df["source"] = "KISTEP"
df["crawled_at"] = datetime.now()

columns = [
    "category",
    "title",
    "author",
    "published_date",
    "detail_url",
    "download_url",
    "page_no",
    "year",
    "source",
    "crawled_at",
]
df = df[columns]

# 한글이 깨지지 않도록 utf-8-sig 로 저장
df.to_csv("kistep_briefs_vibe.csv", index=False, encoding="utf-8-sig")

print("저장 완료: kistep_briefs_vibe.csv")
df.head(10)

저장 완료: kistep_briefs_vibe.csv


In [5]:
#supabase 클라이언트
import os
from dotenv import load_dotenv
from supabase import create_client

load_dotenv()  # .env 파일의 값을 불러온다

url = os.environ.get("SUPABASE_URL")
key = os.environ.get("SUPABASE_SECRET_KEY")

supabase = create_client(url, key)

In [6]:
# supabase에 전달하기 위해 데이터프레임을 JSON 직렬화 가능한 형태로 변환
import numpy as np

df = df.copy()
df["page_no"] = df["page_no"].astype(int)
df["year"] = df["year"].astype(int)
df["crawled_at"] = pd.to_datetime(df["crawled_at"]).dt.strftime("%Y-%m-%d %H:%M:%S.%f")

df = df.replace({np.nan: None})
records = df.to_dict(orient="records")

In [ ]:
#supabase에 저장
response = (
    supabase.table("briefs")
    .upsert(records, on_conflict="detail_url")
    .execute()
)

len(response.data)

30

In [ ]:
#데이터 확인
response = supabase.table("briefs").select("*").execute()
#response = supabase.table("briefs").select("*").eq("year",2026).execute()

rows = response.data

result_df = pd.DataFrame(rows)
result_df.head()


,id,category,title,author,published_date,year,detail_url,download_url,source,page_no,crawled_at,created_at
0,64,정책브리프,"동물실험 대체 전환기, 오가노이드 현황과 시사점 - 주요국 정책 및 국가연구개발사업...","KISTEP 바이오혁신전략팀 심현아 부연구위원, 윤희정 팀장",2026-06-08,2026,https://www.kistep.re.kr/board.es?mid=a1030601...,https://www.kistep.re.kr/boardDownload.es?bid=...,KISTEP,1,2026-06-11 11:01:27.146037,2026-06-11T01:55:14.65976+00:00
1,65,이슈페이퍼,특허 네트워크로 살펴본 글로벌 인공지능 기술 경쟁의 지형 -USPTO 등록특허를 중...,KISTEP 혁신정보분석센터 한웅용 연구위원,2026-05-27,2026,https://www.kistep.re.kr/board.es?mid=a1030601...,https://www.kistep.re.kr/boardDownload.es?bid=...,KISTEP,1,2026-06-11 11:01:27.146037,2026-06-11T01:55:14.65976+00:00
2,66,통계브리프,세계지식재산기구 혁신역량전망 2026 분석,KISTEP 혁신정보분석센터 김선정 전문관리원,2026-05-08,2026,https://www.kistep.re.kr/board.es?mid=a1030601...,https://www.kistep.re.kr/boardDownload.es?bid=...,KISTEP,1,2026-06-11 11:01:27.146037,2026-06-11T01:55:14.65976+00:00
3,67,정책브리프,일본 「제7기 과학기술·혁신 기본계획」 주요 내용 및 시사점,"KISTEP 정책협력팀 이새롬 부연구위원, 김민재 선임전문관리원, 배용국 팀장",2026-04-20,2026,https://www.kistep.re.kr/board.es?mid=a1030601...,https://www.kistep.re.kr/boardDownload.es?bid=...,KISTEP,1,2026-06-11 11:01:27.146037,2026-06-11T01:55:14.65976+00:00
4,68,정책브리프,중국 제15차 5개년 계획의 산업체계 및 과학기술 혁신 분야 주요 내용과 시사점,KISTEP 혁신전략기획센터 전부기 부연구위원,2026-04-08,2026,https://www.kistep.re.kr/board.es?mid=a1030601...,https://www.kistep.re.kr/boardDownload.es?bid=...,KISTEP,1,2026-06-11 11:01:27.146037,2026-06-11T01:55:14.65976+00:00


In [21]:
# id=64 title 끝에 "test" 붙여서 수정
target_id = 64

current = (
    supabase.table("briefs")
    .select("id, title")
    .eq("id", target_id)
    .single()
    .execute()
)

old_title = current.data["title"]
new_title = f"test! {old_title}"

response = (
    supabase.table("briefs")
    .update({"title": new_title})
    .eq("id", target_id)
    .execute()
)

print("변경 전:", old_title)
print("변경 후:", new_title)
response.data

변경 전: 동물실험 대체 전환기, 오가노이드 현황과 시사점 - 주요국 정책 및 국가연구개발사업을 중심으로 -
변경 후: test! 동물실험 대체 전환기, 오가노이드 현황과 시사점 - 주요국 정책 및 국가연구개발사업을 중심으로 -


[{'id': 64,
  'category': '정책브리프',
  'title': 'test! 동물실험 대체 전환기, 오가노이드 현황과 시사점 - 주요국 정책 및 국가연구개발사업을 중심으로 -',
  'author': 'KISTEP 바이오혁신전략팀 심현아 부연구위원, 윤희정 팀장',
  'published_date': '2026-06-08',
  'year': 2026,
  'detail_url': 'https://www.kistep.re.kr/board.es?mid=a10306010000&bid=0031&b_list=10&act=view&list_no=94746&nPage=1&keyField=&orderby=',
  'download_url': 'https://www.kistep.re.kr/boardDownload.es?bid=0031&list_no=94746&seq=1',
  'source': 'KISTEP',
  'page_no': 1,
  'crawled_at': '2026-06-11 11:01:27.146037',
  'created_at': '2026-06-11T01:55:14.65976+00:00'}]

## 4) PDF 다운로드

`files` 폴더를 만들고, 수집 순서대로 `1.pdf`, `2.pdf`, ... 파일명으로 저장합니다.

In [8]:
# import os
# from pathlib import Path

# FILES_DIR = Path("files")
# FILES_DIR.mkdir(exist_ok=True)   # files 폴더 생성

# pdf_filenames = []               # 저장된 PDF 파일명 기록

# for idx, row in df.iterrows():
#     file_no = idx + 1            # 파일명은 1부터 시작
#     filename = f"{file_no}.pdf"
#     filepath = FILES_DIR / filename

#     print(f"[{file_no}/{len(df)}] 다운로드 중: {row['title'][:40]}...")

#     response = requests.get(row["download_url"], headers=headers, timeout=60)
#     response.raise_for_status()

#     with open(filepath, "wb") as f:
#         f.write(response.content)

#     pdf_filenames.append(filename)
#     time.sleep(1)                # 서버 부담을 줄이려 1초 대기

# print(f"\n다운로드 완료: {len(pdf_filenames)}개 PDF → {FILES_DIR}/")
# pd.DataFrame({"title": df["title"], "download_url": df["download_url"], "pdf_filename": pdf_filenames}).head()

## 5) 썸네일 이미지 다운로드

`images` 폴더에 저장하며, 파일명은 HTML `src`의 이미지명을 그대로 사용합니다.
(확장자가 없는 경우 응답 Content-Type 기준으로 `.jpg` 등을 붙입니다.)

In [9]:
# import re
# from urllib.parse import urlparse

# IMAGES_DIR = Path("images")
# IMAGES_DIR.mkdir(exist_ok=True)   # images 폴더 생성


# def get_image_filename(img_src, content_type="image/jpeg"):
#     """img src 속성 기준으로 저장 파일명 생성 (이미지명 그대로)."""
#     src = img_src.lstrip("/")
#     parsed = urlparse("https://x/" + src)
#     basename = parsed.path.rsplit("/", 1)[-1]

#     # 경로에 확장자가 있으면 파일명 그대로 사용 (예: noimage-ebook.jpg)
#     if re.search(r"\.(jpe?g|png|gif|webp)$", basename, re.I):
#         return basename

#     ext_map = {
#         "image/jpeg": ".jpg",
#         "image/jpg": ".jpg",
#         "image/png": ".png",
#         "image/gif": ".gif",
#     }
#     ext = ext_map.get(content_type.split(";")[0].strip().lower(), ".jpg")

#     # 동적 URL은 src 전체를 파일명으로 사용
#     name = src if parsed.query else basename
#     name = re.sub(r'[<>:"/\\|?*]', "_", name)   # Windows에서 사용 불가 문자만 치환
#     if not re.search(r"\.(jpe?g|png|gif|webp)$", name, re.I):
#         name += ext
#     return name


# thumbnail_filenames = []

# for idx, row in df.iterrows():
#     if not row.get("thumbnail_url"):
#         thumbnail_filenames.append(None)
#         continue

#     # src에서 원본 이미지명 추출
#     img_src = urlparse(row["thumbnail_url"]).path
#     if urlparse(row["thumbnail_url"]).query:
#         img_src += "?" + urlparse(row["thumbnail_url"]).query

#     print(f"[{idx + 1}/{len(df)}] 썸네일 다운로드: {row['title'][:40]}...")

#     response = requests.get(row["thumbnail_url"], headers=headers, timeout=60)
#     response.raise_for_status()

#     filename = get_image_filename(img_src, response.headers.get("content-type", "image/jpeg"))
#     filepath = IMAGES_DIR / filename

#     with open(filepath, "wb") as f:
#         f.write(response.content)

#     thumbnail_filenames.append(filename)
#     time.sleep(1)

# print(f"\n다운로드 완료: {sum(1 for x in thumbnail_filenames if x)}개 이미지 → {IMAGES_DIR}/")
# pd.DataFrame({
#     "title": df["title"],
#     "thumbnail_url": [item["thumbnail_url"] for item in items],
#     "thumbnail_filename": thumbnail_filenames,
# }).head()